<a href="https://colab.research.google.com/github/Traiba/aurora-siger/blob/main/analysis-aurora-siger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import time
import random
import pandas as pd
import os
from google.colab import files  # Específico para a Opção A

# ─────────────────────────────────────────────
#  1. UPLOAD DO ARQUIVO (OPÇÃO A)
# ─────────────────────────────────────────────
def carregar_arquivo_local():
    print("Por favor, faça o upload do arquivo 'dataset_telemetria_marte.csv'")
    uploaded = files.upload()
    for nome_arquivo in uploaded.keys():
        print(f'✅ Arquivo "{nome_arquivo}" carregado com sucesso!')
        return nome_arquivo
    return None

# ─────────────────────────────────────────────
#  2. FUNÇÕES DE EXIBIÇÃO E AUXILIARES
# ─────────────────────────────────────────────
def exibir_sucesso():
    print("\n" + "o" * 35)
    print("  UM PEQUENO PASSO PARA UM HOMEM, UM SALTO GIGANTE PARA O BRASIL! 🚀🇧🇷")
    print("  ESTAÇÃO MARCIANA ALCANÇADA - POUSO REALIZADO EM MARTE COM SUCESSO.")
    print("o" * 35 + "\n")
    print("🚀 [MISSÃO MARTE CONCLUÍDA!]")

def exibir_falha(motivos_falha):
    print("\n" + "x" * 35)
    print("  MISSÃO ABORTADA: FALHAS DETECTADAS")
    for motivo in motivos_falha:
        print(f"  - {motivo}")
    print("  PROCEDIMENTOS DE EMERGÊNCIA ATIVADOS. CÁPSULA EJETADA.")
    print("x" * 35 + "\n")
    print("❌ [FALHA NA JORNADA PARA MARTE]")

def realizar_analise_energetica(carga_percentual):
    capacidade_total = 1000  # kWh
    consumo_decolagem = 300  # kWh
    perdas_percentual = 0.05
    energia_disponivel = capacidade_total * (carga_percentual / 100)
    perdas = energia_disponivel * perdas_percentual
    energia_real = energia_disponivel - perdas
    energia_restante = energia_real - consumo_decolagem
    return {
        "disponivel": energia_disponivel,
        "perdas": perdas,
        "real": energia_real,
        "restante": energia_restante,
        "seguro": energia_restante > 0
    }

# ─────────────────────────────────────────────
#  3. LÓGICA DO MINI-GAME
# ─────────────────────────────────────────────
def iniciar_mini_game(telemetria_atual):
    print(f"\n{'='*20} ⚠️ DILEMA NO ESPAÇO PROFUNDO {'='*20}")
    telemetria_pos_minigame = telemetria_atual.copy()

    dilemas = [
        {
            "pergunta": "Uma tempestade solar massiva está vindo! O que fazer?",
            "opcoes": ["1- Ativar escudos (Gasta energia)", "2- Confiar na blindagem (Risco estrutural)"],
            "consequencia1": {"nivel_energia": -15, "msg": "Escudos ativados. Energia drenada."},
            "consequencia2": {"integridade_estrutural": 1, "msg": "Danos estruturais detectados pela radiação."}
        },
        {
            "pergunta": "Vazamento de Oxigênio detectado!",
            "opcoes": ["1- Selar setor (Reduz O2)", "2- Drone de reparo (Gasta bateria)"],
            "consequencia1": {"o2_pct_global": -20, "msg": "Setor selado. O2 reduzido."},
            "consequencia2": {"nivel_energia": -10, "msg": "Drone reparou o vazamento, mas gastou bateria."}
        }
    ]

    dilema = random.choice(dilemas)
    print(f"\n🚨 {dilema['pergunta']}")
    for opt in dilema['opcoes']: print(f"    {opt}")

    escolha = input("Sua escolha (1 ou 2): ")
    res = dilema["consequencia1"] if escolha == "1" else dilema["consequencia2"]
    print(f"\n📢 RESULTADO: {res['msg']}")

    for key, value in res.items():
        if key in telemetria_pos_minigame:
            telemetria_pos_minigame[key] = max(0.0, telemetria_pos_minigame[key] + value)
        elif key == "o2_pct_global":
            global o2_pct_global
            o2_pct_global = max(0.0, o2_pct_global + value)

    return telemetria_pos_minigame

# ─────────────────────────────────────────────
#  4. VERIFICAÇÃO GO/NO-GO
# ─────────────────────────────────────────────
def verificar_go_no_go(telemetria):
    relatorio_falhas = []

    # Critérios simplificados baseados no seu código
    if not (5.0 <= telemetria.get("temperatura_interna", 0) <= 40.0):
        relatorio_falhas.append("Temperatura Interna instável.")
    if telemetria.get("integridade_estrutural", 0) != 0:
        relatorio_falhas.append("Integridade Estrutural comprometida.")
    if telemetria.get("nivel_energia", 0) < 100.0:
        relatorio_falhas.append(f"Energia insuficiente ({telemetria.get('nivel_energia')}%).")
    if telemetria.get("nivel_combustivel", 0) < 100.0:
        relatorio_falhas.append("Tanques de combustível abaixo da capacidade nominal.")

    if not relatorio_falhas:
        return "PRONTO PARA DECOLAR", []
    return "DECOLAGEM ABORTADA", relatorio_falhas

# ─────────────────────────────────────────────
#  5. EXECUÇÃO PRINCIPAL
# ─────────────────────────────────────────────
def simulacao_marte_com_dataset():
    # Passo 1: Upload do arquivo
    nome_csv = carregar_arquivo_local()

    if not nome_csv:
        print("Nenhum arquivo enviado. Simulação encerrada.")
        return

    try:
        df = pd.read_csv(nome_csv)
    except Exception as e:
        print(f"Erro ao ler o arquivo: {e}")
        return

    global o2_pct_global
    o2_pct_global = 100.0

    # Seleciona uma linha aleatória do CSV real
    random_row = df.sample(n=1).iloc[0]
    telemetria_atual = random_row.to_dict()

    print(f"\n{'='*30} DADOS DO DATASET CARREGADO {'='*30}")
    for k, v in telemetria_atual.items():
        print(f"  - {k}: {v}")

    status_inicial, falhas_iniciais = verificar_go_no_go(telemetria_atual)

    if status_inicial == "PRONTO PARA DECOLAR":
        print("\n✅ STATUS INICIAL: TUDO OK! INICIANDO EVENTO ALEATÓRIO...")
        telemetria_final = iniciar_mini_game(telemetria_atual)

        status_pos, falhas_pos = verificar_go_no_go(telemetria_final)
        if o2_pct_global < 50: falhas_pos.append("Oxigênio Crítico!")

        if status_pos == "PRONTO PARA DECOLAR" and not falhas_pos:
            exibir_sucesso()
        else:
            exibir_falha(falhas_pos)
    else:
        exibir_falha(falhas_iniciais)

if __name__ == "__main__":
    simulacao_marte_com_dataset()

Por favor, faça o upload do arquivo 'dataset_telemetria_marte.csv'


Nenhum arquivo enviado. Simulação encerrada.
